In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
dbutils.widgets.text('catalog', 'koine_dev', 'Catalog')
dbutils.widgets.text('schema', 'bronze', 'Schema')
dbutils.widgets.text('table_name', 'fact_sales', 'Table Name')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table_name = dbutils.widgets.get('table_name')

landing_folder_path = 'fact_tables/fact_sales'
if table_name != 'fact_sales':
    landing_folder_path = f'dim_tables/{table_name}'

staging_path = f'/Volumes/koine_dev/staging/staging_data/{landing_folder_path}'
staging_path

## Ingest the Data to Catalog

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS koine_dev.bronze.checkpoints;

In [0]:
checkpoint_path = f'/Volumes/koine_dev/bronze/checkpoints/{table_name}'

def ingest():
    df_stream = (
        spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'parquet')
        .option('cloudFiles.schemaLocation', checkpoint_path)
        .option('cloudFiles.schemaEvolutionMode', "addNewColumns")
        .load(staging_path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias('_file_modified_at'),
            F.current_timestamp().alias("_ingested_at")
        )
    )

    (
        df_stream.writeStream
        .option('checkpointLocation', checkpoint_path)
        .option('mergeSchema', 'true')
        .outputMode('append')
        .trigger(availableNow=True)
        .toTable(f'{catalog}.{schema}.{table_name}')
        .awaitTermination()
    )

In [0]:
ingest()

In [0]:
%sql 
SELECT * FROM koine_dev.bronze.dim_customer ORDER BY last_modified DESC LIMIT 10;

In [0]:
%sql 
SELECT * FROM koine_dev.bronze.dim_customer LIMIT 10 OFFSET 95;